# 00 — Exploratory Data Analysis: StatsBomb Open Data

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)
**Fase:** 0 — Setup y adquisición de datos

## Objetivos de este notebook

1. Descargar / acceder a StatsBomb Open Data.
2. Validar la cobertura de torneos (¿están los Mundiales 2018 y 2022?).
3. Cuantificar el vocabulario potencial: # jugadores, # equipos, # tipos de eventos.
4. Verificar el solapamiento de jugadores entre competiciones — crítico para que los embeddings generalicen.
5. Distribución de resultados (W/D/L) — balance de clases.

## Pregunta epistemológica de fondo

Antes de invertir esfuerzo en arquitectura, queremos contestar: **¿tenemos suficiente data y de la calidad correcta para entrenar un Transformer útil?** Si la cobertura es pobre o el vocabulario está demasiado fragmentado, hay que pivotar.

---
## 1. Setup: Colab + Google Drive

Persistimos los datos en Drive para no perderlos entre sesiones de Colab.

In [ ]:
# Mount Google Drive (skip if running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/d10sformer'
    IN_COLAB = True
except ImportError:
    import os
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

print(f'IN_COLAB: {IN_COLAB}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
import os, sys, json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_RAW = Path(PROJECT_ROOT) / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)
STATSBOMB_PATH = DATA_RAW / 'statsbomb' / 'data'

print(f'DATA_RAW = {DATA_RAW}')
print(f'StatsBomb data path = {STATSBOMB_PATH}')
print(f'Exists: {STATSBOMB_PATH.exists()}')

---
## 2. Descargar StatsBomb Open Data

Solo necesitamos correrlo una vez. El repo pesa ~1.5 GB.

In [ ]:
if not STATSBOMB_PATH.exists():
    print('Cloning StatsBomb Open Data... (~1.5 GB, esto tarda unos minutos)')
    !git clone https://github.com/statsbomb/open-data.git {DATA_RAW}/statsbomb
else:
    print('StatsBomb data already present. Skipping clone.')
    # Opcional: actualizar a la última versión
    # !cd {DATA_RAW}/statsbomb && git pull

---
## 3. Inventario de competiciones disponibles

El archivo `competitions.json` lista todos los torneos cubiertos. Esperamos encontrar:
- Mundial 2018 (Rusia)
- Mundial 2022 (Qatar)
- Eurocopa 2020/2024
- Champions League varias temporadas
- Ligas top (al menos parcial)

In [ ]:
with open(STATSBOMB_PATH / 'competitions.json', 'r') as f:
    competitions = json.load(f)

df_comp = pd.DataFrame(competitions)
print(f'Total competitions x seasons: {len(df_comp)}')
print(f'Unique competitions: {df_comp["competition_name"].nunique()}')
print()
df_comp.groupby('competition_name')['season_name'].apply(list).to_frame()

In [ ]:
# Filtrar Mundiales y torneos de selecciones (lo más cercano al objetivo)
national_team_competitions = df_comp[
    df_comp['competition_name'].str.contains('World Cup|Euro|Copa America|Africa', case=False, na=False)
]
national_team_competitions[['competition_id', 'season_id', 'competition_name', 'season_name']]

---
## 4. Cargar metadatos de partidos

Para cada `(competition_id, season_id)`, existe un archivo `matches/{comp}/{season}.json`.

In [ ]:
def load_matches_for_competition(comp_id, season_id):
    path = STATSBOMB_PATH / 'matches' / str(comp_id) / f'{season_id}.json'
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f)

all_matches = []
for _, row in df_comp.iterrows():
    matches = load_matches_for_competition(row['competition_id'], row['season_id'])
    if matches:
        for m in matches:
            m['competition_name'] = row['competition_name']
            m['season_name'] = row['season_name']
        all_matches.extend(matches)

print(f'Total matches loaded: {len(all_matches)}')
df_matches = pd.json_normalize(all_matches)
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
df_matches.head(3)

In [ ]:
# Distribución temporal
df_matches['year'] = df_matches['match_date'].dt.year
year_counts = df_matches['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
year_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Partidos por año en StatsBomb Open Data')
ax.set_xlabel('Año')
ax.set_ylabel('# partidos')
plt.tight_layout()
plt.show()

print(f'Rango temporal: {df_matches["match_date"].min().date()} a {df_matches["match_date"].max().date()}')

In [ ]:
# Distribución de resultados (W / D / L) desde la óptica del equipo local
df_matches['result'] = np.select(
    [
        df_matches['home_score'] > df_matches['away_score'],
        df_matches['home_score'] == df_matches['away_score'],
    ],
    ['HOME_WIN', 'DRAW'],
    default='AWAY_WIN',
)

result_dist = df_matches['result'].value_counts(normalize=True).sort_index()
print(result_dist)
result_dist.plot(kind='bar', color=['steelblue', 'gray', 'firebrick'], title='Distribución de resultados (perspectiva HOME)')
plt.ylabel('Frecuencia')
plt.show()

**Sanity check:** la frecuencia de empates en datos profesionales suele ser ~25%, victorias locales ~45%, visitantes ~30%. Si nuestro dataset se desvía mucho de esto, vale la pena entender por qué (puede ser por la fuerte presencia de Champions, donde los partidos son más parejos).

---
## 5. Inspección de un partido — estructura de eventos y lineups

Para validar que entendemos el formato antes de diseñar el tokenizer.

In [ ]:
# Tomar un partido emblemático del Mundial 2022 - final Argentina vs Francia
wc_2022 = df_matches[df_matches['competition_name'].str.contains('FIFA World Cup', case=False, na=False)]
if len(wc_2022) > 0:
    # Filtrar finales / partidos importantes
    finals = wc_2022[wc_2022['home_team.home_team_name'].str.contains('Argentina|France', case=False, na=False) &
                     wc_2022['away_team.away_team_name'].str.contains('Argentina|France', case=False, na=False)]
    if len(finals) > 0:
        sample_match_id = finals.iloc[-1]['match_id']
        print(f'Sample match: {finals.iloc[-1]["home_team.home_team_name"]} vs {finals.iloc[-1]["away_team.away_team_name"]}')
    else:
        sample_match_id = wc_2022.iloc[0]['match_id']
else:
    sample_match_id = df_matches.iloc[0]['match_id']

print(f'sample_match_id = {sample_match_id}')

In [ ]:
# Lineup del partido
with open(STATSBOMB_PATH / 'lineups' / f'{sample_match_id}.json', 'r') as f:
    lineup = json.load(f)

print(f'Lineup tiene {len(lineup)} equipos')
for team in lineup:
    print(f"\n=== {team['team_name']} ({len(team['lineup'])} jugadores) ===")
    for player in team['lineup'][:5]:
        print(f"  {player['player_id']:>6} | {player['player_name']:<40} | jersey {player['jersey_number']}")

In [ ]:
# Eventos del partido
with open(STATSBOMB_PATH / 'events' / f'{sample_match_id}.json', 'r') as f:
    events = json.load(f)

print(f'Total eventos en el partido: {len(events)}')
df_events = pd.json_normalize(events)
print(f'Tipos de eventos únicos: {df_events["type.name"].nunique()}')
print()
df_events['type.name'].value_counts().head(20)

**Observación clave:** StatsBomb tiene una granularidad ALTÍSIMA (cada pase, cada movimiento sin balón). Para nuestro tokenizer, vamos a tener que **filtrar** a los eventos relevantes para predicción de resultado: goles, tarjetas, sustituciones, asistencias, tiros, faltas. El resto introduce ruido.

---
## 6. Vocabulario potencial: # jugadores, # equipos, # tipos de eventos

Esto define el tamaño aproximado del vocabulario que va a manejar el Transformer.

In [ ]:
# Equipos únicos
teams_home = set(df_matches['home_team.home_team_name'].dropna().unique())
teams_away = set(df_matches['away_team.away_team_name'].dropna().unique())
all_teams = teams_home | teams_away
print(f'Equipos únicos en el dataset: {len(all_teams)}')

# Sample
print(f'\nEjemplos: {sorted(list(all_teams))[:10]}')

In [ ]:
# Jugadores únicos - esto requiere abrir TODOS los lineups (lento)
# Por ahora, samplear ~200 partidos para una estimación rápida

sample_match_ids = df_matches['match_id'].sample(min(200, len(df_matches)), random_state=42).tolist()

all_players = set()
player_competitions = defaultdict(set)  # jugador -> {competiciones donde aparece}

for mid in sample_match_ids:
    lp = STATSBOMB_PATH / 'lineups' / f'{mid}.json'
    if not lp.exists():
        continue
    with open(lp, 'r') as f:
        teams = json.load(f)
    comp_name = df_matches[df_matches['match_id'] == mid]['competition_name'].iloc[0]
    for team in teams:
        for player in team['lineup']:
            pid = player['player_id']
            all_players.add(pid)
            player_competitions[pid].add(comp_name)

print(f'Jugadores únicos en {len(sample_match_ids)} partidos sampleados: {len(all_players)}')
print(f'\nExtrapolación grosera para todo el dataset: ~{int(len(all_players) * len(df_matches) / len(sample_match_ids))}')

# Solapamiento entre competiciones - CRÍTICO
n_comps_per_player = pd.Series([len(c) for c in player_competitions.values()])
print(f'\n--- Solapamiento de jugadores entre competiciones (en muestra) ---')
print(n_comps_per_player.value_counts().sort_index())

**Hallazgo crítico:** si la mayoría de jugadores aparecen en **1 sola competencia**, el embedding del jugador va a ser pobre — no tiene suficiente "contexto" para aprenderse. Idealmente queremos muchos jugadores que aparezcan en 2+ competiciones (ej. Messi en La Liga + Champions + Mundial).

---
## 7. Conclusiones de Fase 0

Llenar al final de la ejecución:

- [ ] Total de partidos disponibles: ___
- [ ] Cobertura Mundial 2018: ___
- [ ] Cobertura Mundial 2022: ___
- [ ] # equipos únicos: ___
- [ ] # jugadores únicos estimados: ___
- [ ] Distribución W/D/L: ___
- [ ] ¿El vocabulario potencial está en el rango 8k–15k? ___
- [ ] ¿Hay suficiente solapamiento jugador-competiciones? ___

## Verificación de comprensión

1. ¿Por qué consideramos crítico el solapamiento de jugadores entre competiciones para la calidad de los embeddings? Conectalo con la idea de **semántica distribucional** (Firth 1957: *"You shall know a word by the company it keeps"*).
2. Si el dataset tiene 65% victorias locales en partidos de liga pero solo 40% en Mundiales, ¿qué consecuencia tendría entrenar mezclando ambos sin distinguir contexto?